<a href="https://colab.research.google.com/github/AnanyaTyagi/BrainMetShare-3-Benchmarking/blob/main/Brain.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install required libraries

!pip -q install nibabel monai torchmetrics

import os, glob, math, random
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import os

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F


from monai.networks.nets import UNet
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.transforms import Resize
from torchmetrics.classification import BinaryPrecision, BinaryRecall, BinaryAccuracy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 148.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.2/983.2 kB 94.9 MB/s eta 0:00:00


<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.


In [2]:
# Set random seed for reproducibility
#Sets the random seed for Python, NumPy, and PyTorch to ensure reproducible experimental results across multiple runs.

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


In [3]:
# Mount Google Drive and set output directory

from google.colab import drive
OUT_DIR = "/content/output_brainmets"
os.makedirs(OUT_DIR, exist_ok=True)

drive.mount('/content/drive')
DATA_DIR = "/content/drive/MyDrive/brainmetshare-3"
TRAIN_DIR = f"{DATA_DIR}/train"
TEST_DIR  = f"{DATA_DIR}/test"
print("Train dir exists:", os.path.exists(TRAIN_DIR))
print("Test dir exists :", os.path.exists(TEST_DIR))

# Dataset integrity check
def check_split(split_dir, max_cases=5):
    cases = sorted([p for p in glob.glob(os.path.join(split_dir, "*")) if os.path.isdir(p)])
    print(f"\n{split_dir} cases:", len(cases))
    need = ["bravo.nii.gz", "flair.nii.gz", "t1_gd.nii.gz", "t1_pre.nii.gz", "seg.nii.gz"]
    for c in cases[:max_cases]:
        missing = [f for f in need if not os.path.exists(os.path.join(c, f))]
        print(os.path.basename(c), "OK" if not missing else f"Missing: {missing}")

# Run integrity checks on training and test splits
check_split(TRAIN_DIR)
check_split(TEST_DIR)

# Training hyperparameters
IMAGE_SIZE = (256, 256)
BATCH_SIZE = 8
EPOCHS = 20
LR = 1e-3
WEIGHT_DECAY = 1e-5


Mounted at /content/drive
Train dir exists: True
Test dir exists : True

/content/drive/MyDrive/brainmetshare-3/train cases: 105
Mets_005 OK
Mets_010 OK
Mets_011 OK
Mets_013 OK
Mets_014 OK

/content/drive/MyDrive/brainmetshare-3/test cases: 51
Mets_009 Missing: ['seg.nii.gz']
Mets_021 Missing: ['seg.nii.gz']
Mets_025 Missing: ['seg.nii.gz']
Mets_029 Missing: ['seg.nii.gz']
Mets_038 Missing: ['seg.nii.gz']


In [ ]:

# Utility functions for data handling

def load_nii(path: str) -> np.ndarray:
    # Load nifti file into float32 numpy array
    return np.asarray(nib.load(path).get_fdata(), dtype=np.float32)

def normalize_volume(vol: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    # Clip outliers and scale to [0,1]
    v = vol.copy()
    lo, hi = np.percentile(v, 1.0), np.percentile(v, 99.0)
    v = np.clip(v, lo, hi)
    v = (v - v.min()) / (v.max() - v.min() + eps)
    return v

def find_cases(split_dir: str):
    #Find all subject/case directories within a given dataset split

    cases = sorted([p for p in glob.glob(os.path.join(split_dir, "*")) if os.path.isdir(p)])
    if len(cases) == 0:
        raise FileNotFoundError(f"No case folders found in: {split_dir}")
    return cases


# Locate training and test cases

train_cases_all = find_cases(TRAIN_DIR)
test_cases = find_cases(TEST_DIR)

# Filter incomplete test cases

REQUIRED_TEST_FILES = ["t1_pre.nii.gz", "t1_gd.nii.gz", "bravo.nii.gz", "flair.nii.gz"]

def filter_complete_cases(cases):

    #Filter out test cases that do not contain all required MRI modalities.
    good, bad = [], []
    for c in cases:
        missing = [f for f in REQUIRED_TEST_FILES if not os.path.exists(os.path.join(c, f))]
        if missing:
            bad.append((c, missing))
        else:
            good.append(c)
    return good, bad

# Apply filtering to the test set

test_cases_all = test_cases  # keep original list
test_cases, bad_cases = filter_complete_cases(test_cases_all)

print("All test cases:", len(test_cases_all))
print("Complete test cases:", len(test_cases))
print("Skipped incomplete:", len(bad_cases))

# Display a few skipped cases for inspection
for c, miss in bad_cases[:10]:
    print("SKIP:", os.path.basename(c), "missing:", miss)

print("Train cases:", len(train_cases_all))
print("Test cases :", len(test_cases))

# --------------------------------Train / Validation split----------------------------------------
# Shuffle training case list to ensure random selection of validation subjects
random.shuffle(train_cases_all)

# Fraction of training data to use for validation
val_frac = 0.15

# Number of validation cases (at least 1 case)
n_val = max(1, int(len(train_cases_all) * val_frac))

# Split cases into validation and training subsets
val_cases = train_cases_all[:n_val]
train_cases = train_cases_all[n_val:]

print("Train split:", len(train_cases))
print("Val split  :", len(val_cases))

# Target image size after resizing (used by both datasets)
IMAGE_SIZE = (256, 256)


# -----------------Dataset class for TRAIN / VALIDATION (requires seg.nii.gz)---------
# Dataset used for TRAINING and VALIDATION
class BrainMets2DSliceDatasetWithMask(Dataset):

    def __init__(self, cases, image_size=(256,256), training=True,
                 oversample_pos=0.5, max_slices_per_case=96):

        #cases: list of case folder paths, image_size: target size for resizing slices, training: True enables augmentation and positive-slice oversampling, oversample_pos: probability of sampling a slice containing a lesion, max_slices_per_case: limits number of slices per case (faster training)

        self.cases = cases
        self.training = training
        self.oversample_pos = oversample_pos

        # Store slice indices for each case and cache loaded volumes
        self.case_slices = []
        self._cache = {}

        # Precompute slice indices and identify "positive" slices (lesion present)
        for c in self.cases:
            seg_path = os.path.join(c, "seg.nii.gz")
            if not os.path.exists(seg_path):
                raise FileNotFoundError(f"Missing seg.nii.gz in {c}")

            seg = load_nii(seg_path)
            D = seg.shape[-1]                 # number of axial slices
            all_idx = list(range(D))          # all slice indices
            pos_idx = [i for i in all_idx if np.any(seg[..., i] > 0.5)]  # lesion slices

            # keep all positive slices + randomly sample negatives up to max_slices_per_case
            if max_slices_per_case is not None and D > max_slices_per_case:
                keep_pos = pos_idx
                remaining = [i for i in all_idx if i not in set(keep_pos)]
                k = max(0, max_slices_per_case - len(keep_pos))
                keep_neg = random.sample(remaining, k=min(k, len(remaining)))
                all_idx = sorted(list(set(keep_pos + keep_neg)))
                pos_idx = sorted([i for i in all_idx if i in set(keep_pos)])

            self.case_slices.append({"case": c, "all": all_idx, "pos": pos_idx})

        # Total number of training items = total slices across all selected cases
        self.length = sum(len(x["all"]) for x in self.case_slices)

    def __len__(self):
        #Return total number of slice samples in the dataset.
        return self.length

    def _load_case(self, case_path):

        #Load and normalize all 4 modalities and the segmentation mask for one case.Uses caching so repeated slice access is faster.
        if case_path in self._cache:
            return self._cache[case_path]

        # Paths for all MRI modalities + ground truth segmentation mask
        paths = {
            "t1_pre": os.path.join(case_path, "t1_pre.nii.gz"),
            "t1_gd":  os.path.join(case_path, "t1_gd.nii.gz"),
            "flair":  os.path.join(case_path, "flair.nii.gz"),
            "bravo":  os.path.join(case_path, "bravo.nii.gz"),
            "seg":    os.path.join(case_path, "seg.nii.gz"),
        }

        # Ensure all required files exist
        for k, p in paths.items():
            if not os.path.exists(p):
                raise FileNotFoundError(f"Missing {k} in {case_path}")

        # Load + normalize intensities to [0,1]
        t1_pre = normalize_volume(load_nii(paths["t1_pre"]))
        t1_gd  = normalize_volume(load_nii(paths["t1_gd"]))
        flair  = normalize_volume(load_nii(paths["flair"]))
        bravo  = normalize_volume(load_nii(paths["bravo"]))

        # Load binary segmentation mask
        seg = (load_nii(paths["seg"]) > 0.5).astype(np.float32)

        # Sanity check: all modalities and mask must be same shape
        if not (t1_pre.shape == t1_gd.shape == flair.shape == bravo.shape == seg.shape):
            raise ValueError(f"Shape mismatch in {case_path}")

        # Store in cache
        self._cache[case_path] = (t1_pre, t1_gd, flair, bravo, seg)
        return self._cache[case_path]

    def __getitem__(self, idx):

        #Returns one training sample (one 2D slice):
        #Selects a slice index (oversampling lesion slices during training)
        # Stacks modalities into a 4-channel input
        #Extracts the corresponding mask slice
        #Resizes to IMAGE_SIZE
        # Applies data augmentation (training only)

        running = 0

        # Find which case the global idx belongs to
        for info in self.case_slices:
            n = len(info["all"])
            if idx < running + n:
                case_path = info["case"]
                local_idx = idx - running

                # Oversample lesion slices to reduce class imbalance
                if self.training and len(info["pos"]) > 0 and random.random() < self.oversample_pos:
                    z = random.choice(info["pos"])
                else:
                    z = info["all"][local_idx]

                # Load full volumes for the case
                t1_pre, t1_gd, flair, bravo, seg = self._load_case(case_path)

                # Build 4-channel input (4, H, W) and mask (1, H, W)
                x = np.stack([t1_pre[..., z], t1_gd[..., z], flair[..., z], bravo[..., z]], axis=0)
                y = seg[..., z][None, ...]

                # Convert NumPy -> PyTorch tensors
                x = torch.from_numpy(x)
                y = torch.from_numpy(y)

                # Remove unexpected singleton dimensions (robustness)
                x = x.squeeze()
                y = y.squeeze()

                # Ensure channel-first shape (C,H,W)
                if x.ndim == 2:
                    x = x.unsqueeze(0)
                if y.ndim == 2:
                    y = y.unsqueeze(0)

                # Resize slices to standard input size
                # (F.interpolate used because it is robust to shape issues)
                x = F.interpolate(x.unsqueeze(0), size=IMAGE_SIZE, mode="bilinear", align_corners=False).squeeze(0)
                y = F.interpolate(y.unsqueeze(0), size=IMAGE_SIZE, mode="nearest").squeeze(0)

                # Data augmentation applied only during training
                if self.training:
                    # Random horizontal flip
                    if random.random() < 0.5:
                        x = torch.flip(x, dims=[2])
                        y = torch.flip(y, dims=[2])
                    # Random vertical flip
                    if random.random() < 0.5:
                        x = torch.flip(x, dims=[1])
                        y = torch.flip(y, dims=[1])
                    # Random 90-degree rotation
                    k = random.randint(0, 3)
                    if k:
                        x = torch.rot90(x, k, dims=[1, 2])
                        y = torch.rot90(y, k, dims=[1, 2])

                return x, y

            running += n

        raise IndexError("Index out of range")


# -------------------Dataset class for TEST (no seg.nii.gz available)-----------
class BrainMets2DSliceDatasetNoMask(Dataset):
    def __init__(self, cases, image_size=(256,256)):
        self.cases = cases

        # Index stores all slices across all test cases
        self.index = []          # list of (case_path, z)
        self._cache = {}         # cache loaded modalities per case

        # Build slice index list from reference modality (t1_gd)
        for c in self.cases:
            ref_path = os.path.join(c, "t1_gd.nii.gz")
            if not os.path.exists(ref_path):
                raise FileNotFoundError(f"Missing t1_gd.nii.gz in {c}")

            ref = load_nii(ref_path)
            D = ref.shape[-1]
            for z in range(D):
                self.index.append((c, z))

    def __len__(self):
        #Return total number of test slices.
        return len(self.index)

    def _load_case(self, case_path):
        #Load and normalize all 4 modalities for a test case. Uses caching to avoid repeated disk reads.

        if case_path in self._cache:
            return self._cache[case_path]

        paths = {
            "t1_pre": os.path.join(case_path, "t1_pre.nii.gz"),
            "t1_gd":  os.path.join(case_path, "t1_gd.nii.gz"),
            "flair":  os.path.join(case_path, "flair.nii.gz"),
            "bravo":  os.path.join(case_path, "bravo.nii.gz"),
        }

        # Ensure all modalities exist (incomplete test cases should be filtered beforehand)
        for k, p in paths.items():
            if not os.path.exists(p):
                raise FileNotFoundError(f"Missing {k} in {case_path}")

        t1_pre = normalize_volume(load_nii(paths["t1_pre"]))
        t1_gd  = normalize_volume(load_nii(paths["t1_gd"]))
        flair  = normalize_volume(load_nii(paths["flair"]))
        bravo  = normalize_volume(load_nii(paths["bravo"]))

        # Sanity check: modality shapes must match
        if not (t1_pre.shape == t1_gd.shape == flair.shape == bravo.shape):
            raise ValueError(f"Shape mismatch in {case_path}")

        self._cache[case_path] = (t1_pre, t1_gd, flair, bravo)
        return self._cache[case_path]

    def __getitem__(self, idx):
        #Returns one inference sample (one 2D test slice):Loads modalities, Extracts slice z, Stacks into a 4-channel tensor and Resizes to IMAGE_SIZE
        case_path, z = self.index[idx]
        t1_pre, t1_gd, flair, bravo = self._load_case(case_path)

        # Extract 2D slices and squeeze to avoid (H,W,1) shapes
        s_t1_pre = np.squeeze(t1_pre[..., z])
        s_t1_gd  = np.squeeze(t1_gd[..., z])
        s_flair  = np.squeeze(flair[..., z])
        s_bravo  = np.squeeze(bravo[..., z])

        # Store original shape for reference
        orig_shape = s_t1_gd.shape

        # Stack into (4, H, W)
        x = np.stack([s_t1_pre, s_t1_gd, s_flair, s_bravo], axis=0)
        x = torch.from_numpy(x).float()

        # Resize to network input size
        x = F.interpolate(
            x.unsqueeze(0),
            size=IMAGE_SIZE,
            mode="bilinear",
            align_corners=False
        ).squeeze(0)

        return x, case_path, int(z), orig_shape



# Training dataset with data augmentation and positive-slice oversampling
train_ds = BrainMets2DSliceDatasetWithMask(
    train_cases,
    image_size=IMAGE_SIZE,
    training=True,
    oversample_pos=0.5,
    max_slices_per_case=96
)

# Validation dataset (no augmentation, no oversampling)
val_ds = BrainMets2DSliceDatasetWithMask(
    val_cases,
    image_size=IMAGE_SIZE,
    training=False,
    oversample_pos=0.0,
    max_slices_per_case=None
)

# DataLoader for training (shuffled batches)
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

# DataLoader for validation (no shuffling)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# Test dataset (no ground truth masks)
test_ds = BrainMets2DSliceDatasetNoMask(
    test_cases,
    image_size=IMAGE_SIZE
)

# DataLoader for test inference
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# Display dataset sizes
print("Train samples:", len(train_ds))
print("Val samples  :", len(val_ds))
print("Test slices  :", len(test_ds))


# Model Definition (2D U-Net)

# Initialize a 2D U-Net model for binary segmentation
model = UNet(
    spatial_dims=2,              # 2D convolution
    in_channels=4,               # Four MRI modalities as input
    out_channels=1,              # Binary segmentation output
    channels=(32, 64, 128, 256, 512),  # Encoder channel sizes
    strides=(2, 2, 2, 2),        # Downsampling at each level
    num_res_units=2              # Residual units per level
).to(device)

# Loss function: Dice + Binary Cross Entropy
criterion = DiceCELoss(sigmoid=True)

# Optimizer: Adam with weight decay (L2 regularization)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

#-------------------- Evaluation Metrics-----------------------
# Dice similarity coefficient
dice_metric = DiceMetric(include_background=True, reduction="mean")

# Pixel-level precision, recall, and accuracy
precision_metric = BinaryPrecision(threshold=0.5).to(device)
recall_metric    = BinaryRecall(threshold=0.5).to(device)
acc_metric       = BinaryAccuracy(threshold=0.5).to(device)

# Gradient scaler for mixed precision training (GPU only)
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))


#--------------- Validation Evaluation Function------------------------


def evaluate_seg(model, loader):
    #Evaluate performance on validation data.
    model.eval()

    # Reset metric states
    dice_metric.reset()
    precision_metric.reset()
    recall_metric.reset()
    acc_metric.reset()

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            # Forward pass
            logits = model(x)
            probs = torch.sigmoid(logits)

            # Binary prediction
            preds = (probs >= 0.5).float()

            # Update metrics
            dice_metric(y_pred=preds, y=y)
            precision_metric.update(preds.view(-1), y.view(-1))
            recall_metric.update(preds.view(-1), y.view(-1))
            acc_metric.update(preds.view(-1), y.view(-1))

    return (
        dice_metric.aggregate().item(),
        precision_metric.compute().item(),
        recall_metric.compute().item(),
        acc_metric.compute().item()
    )


# ---------------Training Loop-------------------

# Store training history
history = {
    "train_loss": [],
    "val_dice": [],
    "val_precision": [],
    "val_recall": [],
    "val_acc": []
}

best_dice = -1

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    num_batches = 0

    # Iterate over training batches
    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad(set_to_none=True)

        # Mixed precision forward pass
        with torch.amp.autocast("cuda", enabled=(device.type == "cuda")):
            logits = model(x)
            loss = criterion(logits, y)

        # Backpropagation
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()
        num_batches += 1

    # Average training loss
    train_loss = running_loss / max(1, num_batches)

    # Validation evaluation
    val_dice, val_p, val_r, val_a = evaluate_seg(model, val_loader)

    # Store metrics
    history["train_loss"].append(train_loss)
    history["val_dice"].append(val_dice)
    history["val_precision"].append(val_p)
    history["val_recall"].append(val_r)
    history["val_acc"].append(val_a)

    # Print epoch summary
    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Loss {train_loss:.4f} | "
        f"Val Dice {val_dice:.4f} | "
        f"P {val_p:.4f} R {val_r:.4f} Acc {val_a:.4f}"
    )

    # Save best-performing model
    if val_dice > best_dice:
        best_dice = val_dice
        torch.save(
            model.state_dict(),
            os.path.join(OUT_DIR, "best_unet.pth")
        )

print("Best model saved:", os.path.join(OUT_DIR, "best_unet.pth"))


# -----------Training Curves Visualization--------------------

plt.figure()
plt.plot(range(1, EPOCHS + 1), history["train_loss"])
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss vs Epoch")
plt.show()

plt.figure()
plt.plot(range(1, EPOCHS + 1), history["val_dice"])
plt.xlabel("Epoch")
plt.ylabel("Dice Score")
plt.title("Validation Dice vs Epoch")
plt.show()


# ---------------Testingg Inference and Mask Reconstruction----------
# Load best model for inference
best_model = UNet(
    spatial_dims=2,
    in_channels=4,
    out_channels=1,
    channels=(32, 64, 128, 256, 512),
    strides=(2, 2, 2, 2),
    num_res_units=2,
).to(device)

best_model.load_state_dict(
    torch.load(os.path.join(OUT_DIR, "best_unet.pth"), map_location=device)
)
best_model.eval()

#------------ Store predictions for each test case--------------
from collections import defaultdict
pred_store = defaultdict(list)

with torch.no_grad():
    for x, case_path, z, orig_shape in test_loader:
        x = x.to(device)

        # Forward pass
        logits = best_model(x)
        probs = torch.sigmoid(logits).cpu().numpy()

        # Safely unpack batch elements
        case_list = list(case_path)
        z_list = [int(v) for v in z]
        orig_list = list(orig_shape)

        # Store predicted slices
        for case, zi, prob_map, osh in zip(case_list, z_list, probs[:, 0], orig_list):
            pred2d = (prob_map >= 0.5).astype(np.uint8)
            pred_store[case].append((zi, pred2d, osh))


#------------------ Save Predicted Masks as NIfTI Volumes-----------------------

save_dir = os.path.join(OUT_DIR, "test_predictions")
os.makedirs(save_dir, exist_ok=True)

for case, items in pred_store.items():
    items = sorted(items, key=lambda t: t[0])

    # Load reference image for affine and shape
    ref_path = os.path.join(case, "t1_gd.nii.gz")
    ref_img = nib.load(ref_path)
    _, _, D = ref_img.get_fdata().shape

    # Create 3D prediction volume (resized space)
    vol_resized = np.zeros((IMAGE_SIZE[0], IMAGE_SIZE[1], D), dtype=np.uint8)
    for zi, pred2d, _ in items:
        vol_resized[..., zi] = pred2d

    # Save prediction volume
    case_name = os.path.basename(case)
    out_path = os.path.join(save_dir, f"{case_name}_pred_seg_resized.nii.gz")
    nib.save(nib.Nifti1Image(vol_resized, ref_img.affine), out_path)

print("Saved test predictions to:", save_dir)


# --------------------------- Validation Metrics----------------

val_dice, val_p, val_r, val_a = evaluate_seg(best_model, val_loader)
print("\n FINAL VALIDATION METRICS ")
print(f"Dice      : {val_dice:.4f}")
print(f"Precision : {val_p:.4f}")
print(f"Recall    : {val_r:.4f}")
print(f"Accuracy  : {val_a:.4f}")


